# Electronics Sales Pipeline (Source → Bronze → Silver → Gold)

Initial batch load of `electronics_sales_initial.csv` (BATCH_001): read and validate in Bronze with rejection reasons, clean and calculate `net_sales` in Silver, and build the business aggregate `electronics_revenue_by_brand` in Gold.

In [0]:
# STEP 1 (Bronze A): Read and inspect the source file
file_path = "/Volumes/project_1/source/raw_data/electronics_sales_initial.csv"

sales_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path))

print("Schema:")
sales_df.printSchema()

source_count = sales_df.count()
print(f"\nSource count: {source_count}")

print("\nSample rows:")
display(sales_df.limit(10))

In [0]:
# STEP 2 (Bronze B): Create the Bronze DataFrame and add audit columns
from pyspark.sql import functions as F

bronze_df = (sales_df
    .withColumn("source_file", F.lit("electronics_sales_initial.csv"))
    .withColumn("batch_id", F.lit("BATCH_001"))
    .withColumn("ingestion_timestamp", F.current_timestamp()))

print(f"Bronze row count before validation: {bronze_df.count()}")
display(bronze_df.limit(5))

In [0]:
# STEP 3 (Bronze C): Apply validation rules and create a rejection_reason for each rejected row
from pyspark.sql import Window

valid_statuses = ["COMPLETED", "RETURNED", "CANCELLED"]

# each rule is (rejection reason, boolean expression that is True when the rule is VIOLATED)
is_blank = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")

rules = [
    ("sale_id is mandatory", is_blank("sale_id")),
    ("customer_id is mandatory", is_blank("customer_id")),
    ("product_name must not be blank", is_blank("product_name")),
    ("brand must not be blank", is_blank("brand")),
    ("quantity must be greater than 0", F.col("quantity").isNull() | (F.col("quantity") <= 0)),
    ("unit_price must be greater than 0", F.col("unit_price").isNull() | (F.col("unit_price") <= 0)),
    ("discount must be 0 or greater", F.col("discount").isNull() | (F.col("discount") < 0)),
    ("warranty_years must be 0 or greater", F.col("warranty_years").isNull() | (F.col("warranty_years") < 0)),
    ("sale_date must be valid", F.col("sale_date").isNull()),
    ("sale_status must be COMPLETED, RETURNED or CANCELLED",
     ~F.coalesce(F.upper(F.col("sale_status")), F.lit("")).isin(valid_statuses)),
]

# duplicate sale_id check: count rows per sale_id (nulls handled by the mandatory rule)
dup_window = Window.partitionBy("sale_id")
bronze_flagged = bronze_df.withColumn("_dup_count", F.count(F.lit(1)).over(dup_window))

reason_exprs = [F.when(F.coalesce(cond, F.lit(False)), F.lit(name)) for name, cond in rules]
reason_exprs.append(F.when((F.col("sale_id").isNotNull()) & (F.col("_dup_count") > 1), F.lit("duplicate sale_id")))

bronze_flagged = (bronze_flagged
    .withColumn("_reasons", F.array(*reason_exprs))
    .withColumn("rejection_reason",
        F.when(F.size(F.array_compact("_reasons")) == 0, F.lit(None).cast("string"))
         .otherwise(F.array_join(F.array_compact("_reasons"), "; ")))
    .drop("_reasons", "_dup_count"))

bronze_valid = bronze_flagged.filter(F.col("rejection_reason").isNull())
bronze_rejected = bronze_flagged.filter(F.col("rejection_reason").isNotNull())

print(f"Valid rows:   {bronze_valid.count()}")
print(f"Rejected rows: {bronze_rejected.count()}")
print("\nRejected records:")
display(bronze_rejected.select("sale_id", "customer_id", "quantity", "unit_price", "discount", "sale_status", "rejection_reason"))

In [0]:
# STEP 4 (Bronze D): Persist valid Bronze records and rejected records separately
(bronze_valid.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("project_1.bronze.electronics_sales_bronze"))

(bronze_rejected.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("project_1.bronze.electronics_sales_rejected"))

print(f"Valid table row count:    {spark.table('project_1.bronze.electronics_sales_bronze').count()}")
print(f"Rejected table row count: {spark.table('project_1.bronze.electronics_sales_rejected').count()}")

In [0]:
# STEP 5 (Reconciliation): Source Count = Valid Count + Rejected Count
source_count = sales_df.count()
valid_count = spark.table("project_1.bronze.electronics_sales_bronze").count()
rejected_count = spark.table("project_1.bronze.electronics_sales_rejected").count()

recon_status = "PASS" if source_count == valid_count + rejected_count else "FAIL"

recon_df = spark.createDataFrame(
    [(source_count, valid_count, rejected_count, valid_count + rejected_count, recon_status)],
    ["source_count", "valid_count", "rejected_count", "valid_plus_rejected", "status"])
display(recon_df)

print("Rejection reason breakdown:")
display(spark.table("project_1.bronze.electronics_sales_rejected").groupBy("rejection_reason").count().orderBy(F.desc("count")))

In [0]:
# STEP 6 (Silver A): Clean, cast, calculate net_sales — one row per unique sale_id
silver_df = (spark.table("project_1.bronze.electronics_sales_bronze")
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("brand", F.upper("brand"))
    .withColumn("store_city", F.upper("store_city"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount", F.col("discount").cast("double"))
    .withColumn("warranty_years", F.col("warranty_years").cast("int"))
    .withColumn("net_sales", (F.col("quantity") * F.col("unit_price")) - F.col("discount"))
    .drop("rejection_reason")
    .dropDuplicates(["sale_id"]))

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("project_1.silver.electronics_sales_silver"))

print(f"Silver row count: {spark.table('project_1.silver.electronics_sales_silver').count()}")
display(spark.table("project_1.silver.electronics_sales_silver").limit(5))

In [0]:
# STEP 7 (Silver B): Confirm rejected records did NOT enter Silver
silver_tbl = spark.table("project_1.silver.electronics_sales_silver")
rejected_tbl = spark.table("project_1.bronze.electronics_sales_rejected")

rejected_in_silver = (silver_tbl
    .join(rejected_tbl.filter(F.col("sale_id").isNotNull()), "sale_id", "inner")
    .count())

print(f"Rejected records found in Silver: {rejected_in_silver}")
print("PASS: no rejected records entered Silver" if rejected_in_silver == 0 else "FAIL: rejected records leaked into Silver")

In [0]:
# STEP 8 (Gold): Build electronics_revenue_by_brand — one row per brand
gold_df = (spark.table("project_1.silver.electronics_sales_silver")
    .groupBy("brand")
    .agg(
        F.count("sale_id").alias("total_sales_count"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum("net_sales"), 2).alias("total_revenue"),
        F.round(F.avg("net_sales"), 2).alias("avg_sale_value")))

(gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("project_1.gold.electronics_revenue_by_brand"))

display(spark.table("project_1.gold.electronics_revenue_by_brand").orderBy(F.desc("total_revenue")))

In [0]:
# STEP 9 (Gold): Which brand generated the highest total revenue?
top_brand = spark.table("project_1.gold.electronics_revenue_by_brand").orderBy(F.desc("total_revenue")).first()

print(f"Brand with highest total revenue: {top_brand['brand']}")
print(f"  Total revenue:     {top_brand['total_revenue']}")
print(f"  Total sales count: {top_brand['total_sales_count']}")
print(f"  Total quantity:    {top_brand['total_quantity']}")
print(f"  Avg sale value:    {top_brand['avg_sale_value']}")